In [ ]:
from os import name
from turtledemo.penrose import start

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bokeh.layouts import column
from openpyxl.styles.builtins import total
from statsmodels.tsa.base.datetools import dates_from_range

    sns.set_theme(style="whitegrid",font_scale = 1.2)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [27]:
df = pd.read_parquet('user_behavior_cleaned_dataset.parquet')
df.head()

,user_id,item_id,category_id,behavior_type,timestamp,date
0,1000169,1328010,959452,pv,2017-09-11 08:16:39,2017-09-11
1,112815,2489903,1487591,pv,2017-11-06 10:44:52,2017-11-06
2,108233,4336503,3531700,pv,2017-11-10 13:33:35,2017-11-10
3,114678,3662082,1080785,pv,2017-11-12 06:19:40,2017-11-12
4,12775,2897780,886203,pv,2017-11-12 22:37:24,2017-11-13


分析被购买最多次数的商品类目

In [ ]:
# 按照品类分组后找到最后欢迎的商品类目前十
df_buy = df[df['behavior_type'] == 'buy'].groupby('category_id').size().reset_index(name='buy_count')
buy_top10 = df_buy.sort_values(by='buy_count',ascending = False).head(10)
print(f"购买量前十{buy_top10}")

分析被浏览次数最多的商品

In [ ]:
#按照浏览量分组查询浏览次数前十
df_pv = df[df['behavior_type'] == 'pv'].groupby('item_id').size().reset_index(name='pv_count')
pv_top10 = df_pv.sort_values(by='pv_count',ascending = False).head(10)
print(f"浏览量前十{pv_top10}")

复购率

In [ ]:
#先计算每个用户的下单总量
user_buy_count = df[df['behavior_type'] == 'buy'].groupby(['user_id']).size().reset_index(name='buy_count')
#再计算购买次数大于等于2的用户
repurchase_user = user_buy_count[user_buy_count['buy_count'] >= 2]
# 计算复购率
repeat_purchase = len(repurchase_user)*100/len(user_buy_count)

print(f"复购率为{repeat_purchase:.2f}%")

复购率显著高于行业正常值,可能是数据集中仅包含限定日期内的活跃用户,并不包含流失用户,只能反映高活跃用户的购买粘性,不具有大盘代表性

重新计算可能合理的复购率,只计算核心日期前(11月25日)才开始购买第一次的用户,并且计算限定窗口为七天计算复购率

In [38]:
df1 = df[df['behavior_type'] == 'buy'].sort_values(by=['user_id','date'])
df1['buy_frequency'] = df1.groupby(['user_id']).cumcount()+1
start_date = '2017-11-25'

df_first_buy = df1[(df1['buy_frequency'] == 1) & (df1['date'] >= start_date)][['user_id','date']].rename(columns = {'date':'first_date'})
df_second_buy = df1[df1['buy_frequency'] == 2][['user_id','date']].rename(columns = {'date':'second_date'})

merged = pd.merge(df_first_buy,df_second_buy,on=['user_id'],how='left')

merged['days_gap'] = (merged['second_date'] - merged['first_date']).dt.days

total_new_user = len(merged)
new_repurchase_user = merged[merged['days_gap'] <= 7]
new_repeat_purchase = len(new_repurchase_user)*100/len(merged)

print(f"新用户总数: {total_new_user}")
print(f"7天内复购人数: {len(new_repurchase_user)}")
print(f"保守复购率: {new_repeat_purchase:.2f}%")

新用户总数: 7000
7天内复购人数: 4583
保守复购率: 65.47%
